# Offline Evaluation: Baseline vs Test Model

**Baseline**: `unity-ads-dd-ds-dev-prd.bulk_eval.eval-gfx8xhdufe`  
**Test** (no creative features): `unity-ads-dd-ds-dev-prd.bulk_eval.eval-svsm2opjod`

Metrics: BCE, AUC, GINI, Model Bias, breakdown by `prob_sdk_event_name`, calibration curve.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from google.cloud import bigquery
from sklearn.metrics import roc_auc_score, log_loss

client = bigquery.Client(project="unity-ads-dd-ds-dev-prd")

## 1. Load feature mappings (integer → event name)

In [ ]:
BASELINE_MAPPING_PATH = "v11_cpe_lc_feature_mapping.json"
TEST_MAPPING_PATH     = "v11_cpe_lc_v2_feature_mapping.json"

def load_event_name_mapping(path):
    """Return {integer: event_name_string} from feature mapping JSON."""
    with open(path) as f:
        full_map = json.load(f)
    raw = full_map["prob_sdk_event_name"]  # {str: int}
    return {v: k for k, v in raw.items()}  # invert → {int: str}

baseline_event_map = load_event_name_mapping(BASELINE_MAPPING_PATH)
test_event_map     = load_event_name_mapping(TEST_MAPPING_PATH)

print(f"Baseline mapping: {len(baseline_event_map)} entries")
print(f"Test mapping:     {len(test_event_map)} entries")

## 2. Load evaluation data from BigQuery

In [ ]:
BASELINE_TABLE = "unity-ads-dd-ds-dev-prd.bulk_eval.`eval-gfx8xhdufe`"
TEST_TABLE     = "unity-ads-dd-ds-dev-prd.bulk_eval.`eval-svsm2opjod`"

QUERY = """
    SELECT
        model_pred,
        label,
        prob_sdk_event_name_label,
        prob_sdk_event_name
    FROM {table}
    WHERE model_pred IS NOT NULL
      AND prob_sdk_event_name_label IS NOT NULL
"""

print("Loading baseline...")
df_base = client.query(QUERY.format(table=BASELINE_TABLE)).to_dataframe()
print(f"  {len(df_base):,} rows")

print("Loading test...")
df_test = client.query(QUERY.format(table=TEST_TABLE)).to_dataframe()
print(f"  {len(df_test):,} rows")

In [ ]:
# Map integer → event name string
df_base["event_name"] = df_base["prob_sdk_event_name"].map(baseline_event_map)
df_test["event_name"] = df_test["prob_sdk_event_name"].map(test_event_map)

print("Baseline unmapped:", df_base["event_name"].isna().sum())
print("Test unmapped:    ", df_test["event_name"].isna().sum())

df_base.head(3)

## 3. Overall metrics: BCE, AUC, GINI, Model Bias

In [ ]:
def compute_metrics(df, label_col="prob_sdk_event_name_label", pred_col="model_pred", name="model"):
    y_true = df[label_col].values
    y_pred = df[pred_col].values
    bce  = log_loss(y_true, y_pred)
    auc  = roc_auc_score(y_true, y_pred)
    gini = 2 * auc - 1
    bias = (y_pred.sum() / y_true.sum()) - 1
    return {"Model": name, "BCE": bce, "AUC": auc, "GINI": gini, "Bias": bias}

base_m = compute_metrics(df_base, name="Baseline")
test_m = compute_metrics(df_test,  name="Test (no creative)")

overall = pd.DataFrame([base_m, test_m]).set_index("Model")

abs_delta = overall.loc["Test (no creative)"] - overall.loc["Baseline"]
rel_delta = abs_delta / overall.loc["Baseline"] * 100  # %

# Build display table with string formatting per row type
display_rows = {}
for idx in overall.index:
    display_rows[idx] = {c: f"{overall.loc[idx, c]:.6f}" for c in overall.columns}
display_rows["Δ (abs)"] = {c: f"{abs_delta[c]:+.6f}" for c in overall.columns}
display_rows["Δ% (rel)"] = {c: f"{rel_delta[c]:+.4f}%" for c in overall.columns}

display_df = pd.DataFrame(display_rows).T
display_df.index.name = "Model"

# Highlight delta rows
def highlight_delta(row):
    if row.name.startswith("Δ"):
        return ["background-color: #2a2a3a; font-weight: bold"] * len(row)
    return [""] * len(row)

display_df.style.apply(highlight_delta, axis=1)

## 4. Performance breakdown by `prob_sdk_event_name`

In [ ]:
def metrics_by_event(df, label_col="prob_sdk_event_name_label", pred_col="model_pred", min_samples=100):
    rows = []
    for event, grp in df.groupby("event_name"):
        if len(grp) < min_samples:
            continue
        y_true = grp[label_col].values
        y_pred = grp[pred_col].values
        n_classes = len(np.unique(y_true))
        if n_classes < 2:
            auc = gini = bce = np.nan
        else:
            auc  = roc_auc_score(y_true, y_pred)
            gini = 2 * auc - 1
            bce  = log_loss(y_true, y_pred)
        rows.append({
            "event_name": event,
            "n": len(grp),
            "BCE":  bce,
            "AUC":  auc,
            "GINI": gini,
            "Bias": (y_pred.sum() / y_true.sum()) - 1 if y_true.sum() > 0 else np.nan,
        })
    return pd.DataFrame(rows).set_index("event_name")

ev_base = metrics_by_event(df_base)
ev_test = metrics_by_event(df_test)

ev_compare = ev_base[["n", "BCE", "AUC", "GINI", "Bias"]].join(
    ev_test[["n", "BCE", "AUC", "GINI", "Bias"]],
    lsuffix="_base", rsuffix="_test",
    how="outer"
)
ev_compare["GINI_delta"] = ev_compare["GINI_test"] - ev_compare["GINI_base"]
ev_compare["BCE_delta"]  = ev_compare["BCE_test"]  - ev_compare["BCE_base"]
ev_compare["Bias_delta"] = ev_compare["Bias_test"] - ev_compare["Bias_base"]

ev_compare_sorted = ev_compare.sort_values("n_base", ascending=False)
ev_compare_sorted.head(30).style.format("{:.4f}", na_rep="—", subset=ev_compare_sorted.columns[1:])

In [ ]:
# Summary: events where GINI changed the most
top_gini_change = ev_compare.dropna(subset=["GINI_delta"]).sort_values("GINI_delta")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, data, title in [
    (axes[0], top_gini_change.head(20), "Top 20 GINI Decrease (Test vs Baseline)"),
    (axes[1], top_gini_change.tail(20), "Top 20 GINI Increase (Test vs Baseline)"),
]:
    colors = ["red" if v < 0 else "green" for v in data["GINI_delta"]]
    ax.barh(data.index, data["GINI_delta"], color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel("GINI delta (Test - Baseline)")
    ax.tick_params(axis="y", labelsize=7)

plt.tight_layout()
plt.show()

## 5. Calibration curve: prediction vs label bucketed into 10 bins

In [ ]:
def calibration_curve_data(df, n_bins=10, pred_col="model_pred", label_col="prob_sdk_event_name_label"):
    """Bucket by equal-frequency bins of prediction, return mean pred & mean label per bin."""
    df = df.copy()
    df["bin"] = pd.qcut(df[pred_col], q=n_bins, labels=False, duplicates="drop")
    agg = df.groupby("bin").agg(
        mean_pred=(pred_col, "mean"),
        mean_label=(label_col, "mean"),
        count=(pred_col, "count"),
    ).reset_index()
    return agg

cal_base = calibration_curve_data(df_base)
cal_test = calibration_curve_data(df_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, cal, title in [
    (axes[0], cal_base, "Baseline"),
    (axes[1], cal_test, "Test (no creative features)"),
]:
    ax.plot(cal["bin"], cal["mean_label"], "o-", label="Label", color="steelblue")
    ax.plot(cal["bin"], cal["mean_pred"],  "o-", label="Prediction", color="darkorange")
    ax.set_title(title)
    ax.set_xlabel("Bin")
    ax.set_ylabel("Average Value")
    ax.legend()
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.suptitle("Calibration Curve (10 equal-frequency bins)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
def plot_calibration_bucket(cal, title, ax_bar, ax_gap):
    """
    ax_bar : grouped bar chart of mean label vs mean prediction per bin
             with sample count on secondary y-axis
    ax_gap : bar chart of (prediction - label) gap per bin
    """
    bins   = cal["bin"].values
    labels = cal["mean_label"].values
    preds  = cal["mean_pred"].values
    counts = cal["count"].values
    gap    = preds - labels

    x     = np.arange(len(bins))
    width = 0.35

    # --- top panel: grouped bars + count line ---
    bars1 = ax_bar.bar(x - width/2, labels, width, label="Label",      color="steelblue",  alpha=0.85)
    bars2 = ax_bar.bar(x + width/2, preds,  width, label="Prediction", color="darkorange", alpha=0.85)

    ax_bar.set_title(title, fontsize=11)
    ax_bar.set_ylabel("Average Value")
    ax_bar.set_xticks(x)
    ax_bar.set_xticklabels([f"B{int(b)}" for b in bins])
    ax_bar.legend(loc="upper left", fontsize=8)

    ax2 = ax_bar.twinx()
    ax2.plot(x, counts, "k--o", markersize=4, linewidth=1, label="Count", alpha=0.5)
    ax2.set_ylabel("Sample count", fontsize=8, color="grey")
    ax2.tick_params(axis="y", labelcolor="grey", labelsize=7)
    ax2.legend(loc="upper right", fontsize=7)

    # --- bottom panel: gap bars ---
    colors_gap = ["tomato" if g > 0 else "steelblue" for g in gap]
    ax_gap.bar(x, gap, color=colors_gap, alpha=0.85)
    ax_gap.axhline(0, color="black", linewidth=0.8)
    ax_gap.set_ylabel("Pred − Label", fontsize=9)
    ax_gap.set_xlabel("Bin")
    ax_gap.set_xticks(x)
    ax_gap.set_xticklabels([f"B{int(b)}" for b in bins])

    # annotate gap values
    for i, g in enumerate(gap):
        ax_gap.text(i, g + (0.0002 if g >= 0 else -0.0004),
                    f"{g:+.4f}", ha="center", va="bottom" if g >= 0 else "top",
                    fontsize=6.5, color="black")


fig, axes = plt.subplots(2, 2, figsize=(16, 10),
                          gridspec_kw={"height_ratios": [3, 1.2]})

plot_calibration_bucket(cal_base, "Baseline",                  axes[0][0], axes[1][0])
plot_calibration_bucket(cal_test, "Test (no creative features)", axes[0][1], axes[1][1])

fig.suptitle("Calibration Bucket Plot — Prediction vs Label (10 equal-frequency bins)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Overlay both models on one plot for direct comparison
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(cal_base["bin"], cal_base["mean_label"],  "o-",  label="Label (Baseline)",        color="steelblue",   linestyle="solid")
ax.plot(cal_base["bin"], cal_base["mean_pred"],   "o--", label="Prediction (Baseline)",   color="steelblue",   linestyle="dashed")
ax.plot(cal_test["bin"], cal_test["mean_label"],  "s-",  label="Label (Test)",            color="darkorange",  linestyle="solid")
ax.plot(cal_test["bin"], cal_test["mean_pred"],   "s--", label="Prediction (Test)",       color="darkorange",  linestyle="dashed")

ax.set_title("Calibration Curve Overlay — Baseline vs Test")
ax.set_xlabel("Bin")
ax.set_ylabel("Average Value")
ax.legend()
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

In [ ]:
# Per-event calibration curves (top N events by sample count)
TOP_N_EVENTS = 9

top_events = (
    df_base.groupby("event_name")["model_pred"]
    .count()
    .sort_values(ascending=False)
    .head(TOP_N_EVENTS)
    .index.tolist()
)

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, event in enumerate(top_events):
    ax = axes[i]
    sub_base = df_base[df_base["event_name"] == event]
    sub_test = df_test[df_test["event_name"] == event]

    for sub, label_prefix, color in [
        (sub_base, "Baseline", "steelblue"),
        (sub_test, "Test",     "darkorange"),
    ]:
        if len(sub) < 20:
            continue
        cal = calibration_curve_data(sub)
        ax.plot(cal["bin"], cal["mean_label"], "o-",  label=f"Label ({label_prefix})",      color=color, alpha=0.7)
        ax.plot(cal["bin"], cal["mean_pred"],  "o--", label=f"Prediction ({label_prefix})", color=color, alpha=0.7, linestyle="dashed")

    ax.set_title(event, fontsize=8)
    ax.set_xlabel("Bin", fontsize=7)
    ax.set_ylabel("Avg Value", fontsize=7)
    ax.legend(fontsize=6)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(f"Per-Event Calibration Curves — Top {TOP_N_EVENTS} Events", fontsize=13)
plt.tight_layout()
plt.show()